# Procesamiento de lenguaje natural aplicado

## Fundamentos de análisis de sentimientos y emociones 

In [4]:
import pandas as pd
import torch
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns 

In [5]:
# datos 
data = [
    "La cámara del teléfono es excelente, pero la batería dura muy poco",
    "El envío fue rápido y el producto llegó en buen estado",
    "La atención al cliente fue lenta y poco clara",
    "El precio es alto, aunque la calidad del producto es muy buena",
    "The camera quality is amazing, buy the battery life is disappointing",
    "Customer support was helpful and fast",
    "The delivery was late and the packaging was damaged",
    "The price is reasonable and the performance is excellent"
    
]

In [8]:
df = pd.DataFrame({'texto': data})

In [9]:
df

,texto
0,"La cámara del teléfono es excelente, pero la b..."
1,El envío fue rápido y el producto llegó en bue...
2,La atención al cliente fue lenta y poco clara
3,"El precio es alto, aunque la calidad del produ..."
4,"The camera quality is amazing, buy the battery..."
5,Customer support was helpful and fast
6,The delivery was late and the packaging was da...
7,The price is reasonable and the performance is...


In [14]:
#Clasificador de sentimientos
sentiment_classifer = pipeline(
    task= "sentiment-analysis",
    model= "nlptown/bert-base-multilingual-uncased-sentiment"    
)

df['sentimiento_general'] = df['texto'].apply(lambda x: sentiment_classifer(x)[0]['label'])

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [15]:
df

,texto,sentimiento_general
0,"La cámara del teléfono es excelente, pero la b...",3 stars
1,El envío fue rápido y el producto llegó en bue...,5 stars
2,La atención al cliente fue lenta y poco clara,2 stars
3,"El precio es alto, aunque la calidad del produ...",3 stars
4,"The camera quality is amazing, buy the battery...",3 stars
5,Customer support was helpful and fast,4 stars
6,The delivery was late and the packaging was da...,1 star
7,The price is reasonable and the performance is...,5 stars


In [17]:
aspects = {
    "camera":["camera","cámara"],
    "battery": ["battery", "batería"],
    "delivery": ["delivery", "envío"],
    "support": ["support", "soporte", "atención", "customer support"],
    "price": ["price", "precio"],
    "quality": ["quality", "calidad"],
    "performance": ["Performance", "rendimento"]
}

In [20]:
#Pipeline ---> Verificar bien como funciona esto check how works that shit
aspect_sentiment_model = pipeline(task = "sentiment-analysis", model= "distilbert-base-uncased-finetuned-sst-2-english")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [22]:
aspect_results = []

for text in df['texto']:
    text_lower = text.lower()
    for aspect, keyword, in aspects.items():
        if any(k in text_lower for k in keyword):
            sentiment = aspect_sentiment_model(text)[0]
            aspect_results.append({
                "texto": text,
                "aspecto": aspect,
                "sentimiento": sentiment['label'],
                "confianza": round(sentiment['score'],3)
            })

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [23]:
aspect_df = pd.DataFrame(aspect_results)
aspect_df

,texto,aspecto,sentimiento,confianza
0,"La cámara del teléfono es excelente, pero la b...",camera,NEGATIVE,0.919
1,"La cámara del teléfono es excelente, pero la b...",battery,NEGATIVE,0.919
2,El envío fue rápido y el producto llegó en bue...,delivery,POSITIVE,0.924
3,La atención al cliente fue lenta y poco clara,support,NEGATIVE,0.889
4,"El precio es alto, aunque la calidad del produ...",price,NEGATIVE,0.914
5,"El precio es alto, aunque la calidad del produ...",quality,NEGATIVE,0.914
6,"The camera quality is amazing, buy the battery...",camera,POSITIVE,0.809
7,"The camera quality is amazing, buy the battery...",battery,POSITIVE,0.809
8,"The camera quality is amazing, buy the battery...",quality,POSITIVE,0.809
9,Customer support was helpful and fast,support,POSITIVE,1.000


In [24]:
#EJemplo
print('Ejemplo de diferencia entre enfoques')

#Seleccionar una fila de ejemplo

example = aspect_df.iloc[1]

print("Texto")
print(example['texto'])

print("Aspecto analizado")
print(example['aspecto'])

print('Sentimiento por aspecto')
print(example['sentimiento'])



Ejemplo de diferencia entre enfoques
Texto
La cámara del teléfono es excelente, pero la batería dura muy poco
Aspecto analizado
battery
Sentimiento por aspecto
NEGATIVE


---

# Esto es aparte

In [25]:
import spacy
nlp = spacy.load('es_core_news_sm')
doc = nlp("Los gatos están durmiendo en el sofá.")
for token in doc:
    print(f"Token: {token.text}, Lemma: {token.lemma_}")

OSError: [E050] Can't find model 'es_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

Limpieza mediante remoción de stopwords
Las stopwords son palabras comunes que aportan poco significado analítico. spaCy incluye una lista predefinida que podemos usar para filtrarlas:

In [26]:
clean_tokens = [token for token in doc if not token.is_stop and not token.is_punct]

NameError: name 'doc' is not defined

Integración de modelos preentrenados
Los modelos preentrenados permiten añadir análisis avanzados sin necesidad de entrenar desde cero. Por ejemplo, para análisis de entidades nombradas o vectores de palabras:

In [ ]:
for ent in doc.ents:
    print(ent.text, ent.label_)